# Phase 1 — Transformer Training (Colab / Kaggle)

Steps:
1. Mount Drive and set working directory.
2. Clone / pull the repo.
3. Confirm `story.txt` is present (copy from Drive if needed).
4. Install dependencies.
5. Build SentencePiece BPE tokenizer.
6. Encode corpus → `data/train.bin`, `data/val.bin`.
7. Train the Transformer.
8. Copy checkpoint + tokenizer to Drive.
9. Quick sanity test.

**Before running:** Runtime → Change runtime type → GPU (T4 or better).

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/uni_bitirme'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive directory:', DRIVE_DIR)

In [ ]:
# Cell 1 — Clone / update repo
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'  # <-- fill in
REPO_DIR = '/content/uni_bitirme'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls -la

In [ ]:
# Cell 2 — Make sure story.txt is here
# story.txt is gitignored (514 MB). Copy from Drive if not already present.
STORY_DRIVE = f'{DRIVE_DIR}/story.txt'

if not os.path.exists('story.txt'):
    if os.path.exists(STORY_DRIVE):
        print('Copying story.txt from Drive...')
        !cp {STORY_DRIVE} story.txt
    else:
        raise FileNotFoundError(
            f'story.txt not found.\n'
            f'Upload it to {STORY_DRIVE} on Drive, then re-run this cell.'
        )

!ls -lh story.txt

In [ ]:
# Cell 3 — Install dependencies
!pip install -q sentencepiece tqdm
import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# Cell 4 — Build SentencePiece BPE tokenizer
# Reads all 514 MB of story.txt, cleans Wiki @-@ markers,
# injects TR001..TR017 punctuation tokens, trains 16k vocab BPE.
# Takes ~5-10 minutes. Outputs: tokenizer/spm.model, tokenizer/spm.vocab
!python tokenizer/build_tokenizer.py --input story.txt --vocab_size 16000

In [ ]:
# Cell 5 — Encode corpus
# Streams story.txt through the tokenizer → uint16 binary files.
# data/train.bin = 95%, data/val.bin = 5%
!python tokenizer/encode_corpus.py \
    --input story.txt \
    --tokenizer tokenizer/spm.model \
    --out_dir data \
    --val_fraction 0.05
!ls -lh data/

In [ ]:
# Cell 6 — Train (20k steps ≈ 3-6 hours on T4)
# Checkpoints saved to checkpoints/transformer.pt (best val loss)
# and checkpoints/transformer_step_*.pt every 2000 steps.
# You can interrupt and resume with --resume checkpoints/transformer_step_XXXX.pt
!python transformer/train.py \
    --data_dir data \
    --tokenizer tokenizer/spm.model \
    --steps 20000 \
    --batch_size 32 \
    --block_size 256 \
    --eval_interval 500 \
    --save_interval 2000 \
    --warmup_steps 500 \
    --lr 3e-4

In [ ]:
# Cell 7 — Back up to Drive (won't be lost if session expires)
!cp checkpoints/transformer.pt {DRIVE_DIR}/transformer.pt
!cp tokenizer/spm.model {DRIVE_DIR}/spm.model
!cp tokenizer/spm.vocab {DRIVE_DIR}/spm.vocab
print('Saved to Drive. Download transformer.pt and spm.model to your local repo.')
print('  -> Place transformer.pt at: checkpoints/transformer.pt')
print('  -> Place spm.model at:      tokenizer/spm.model')

In [ ]:
# Cell 8 — Sanity test: generate a sentence from a few prompts
import sys; sys.path.insert(0, '.')
from transformer.sample import TransformerEngine

engine = TransformerEngine('checkpoints/transformer.pt', 'tokenizer/spm.model')

prompts = [
    'The history of',
    'In the early years',
    'Scientists discovered that',
    'The war began when',
]

print('=== Sentence completions ===')
for p in prompts:
    out = engine.generate_until_sentence_end(p, max_new_tokens=80)
    print(f'  {p}{out}')
    print()